In [ ]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np

# Get inference device
output_filename = "/home/tong/recordings/PLYs/scripps924/scripps924_5.ply" 
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model - This requires internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# Load and preprocess images from a folder or list of paths

# images = ["/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106044.png"]
# ...

# 1. Provide a list of multiple image paths
images = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000000.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000050.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000100.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000150.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    
]

views = load_images(images)

# Run inference (this will process all images in the list)
predictions1 = model.infer(
    views,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





geometries = []
camera_positions = []


# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
# geometries1.append(world_frame)

# PLY = o3d.geometry.PointCloud()

last_pose = None
for i, pred1 in enumerate(predictions1):
    
    points_cam = pred1["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(points_cam)
    pcd1.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    
    
    # pcd1.transform(camera_pose)

    # camera_center = camera_pose[:3, 3]
    # camera_positions.append(camera_center)
    
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    
    geometries.append(pcd1)
    geometries.append(camera_frame)


    # PLY += pcd1

    if i == len(predictions1) - 1:
        last_pose = pred1["camera_poses"].squeeze().cpu().numpy()  # keep batch dim
         

if len(camera_positions) > 1:
    # Define the points for the line set
    line_points = o3d.utility.Vector3dVector(camera_positions)
    # Define which points to connect (0->1, 1->2, etc.)
    line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
    lines = o3d.utility.Vector2iVector(line_indices)
    
    # Create the LineSet object
    camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
    # Set the color of the path to red
    camera_path.paint_uniform_color([1, 0, 0])
    
    # Add the path to our list of things to draw
    geometries.append(camera_path)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  # Inverts Z
    [0,  0,  0,  1]
])

for geometry in geometries:
    geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
# print(f"Displaying combined scene with {len(predictions1)} point clouds...")
# o3d.visualization.draw_geometries(geometries1)
# PLY.transform(transform_matrix)
# o3d.io.write_point_cloud(output_filename, PLY)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


PointCloud with 1015280 points.

In [14]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np

# Get inference device
output_filename = "/home/tong/recordings/PLYs/maritime1.ply" 
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# 1. Provide a list of multiple image paths
images = [
    "/home/tong/recordings/maritime/frame_000406.jpg",
    "/home/tong/recordings/maritime/frame_000454.jpg",
    "/home/tong/recordings/maritime/frame_000501.jpg",
    "/home/tong/recordings/maritime/frame_000541.jpg",
    "/home/tong/recordings/maritime/frame_000564.jpg",
]

views = load_images(images)

# Run inference
predictions1 = model.infer(
    views,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

geometries = []
camera_positions = []

# --- SET YOUR FILTER DISTANCE HERE ---
# This will remove points farther than this distance (in meters/units)
# from the camera's origin.
MAX_FILTER_DISTANCE = 10.0 
print(f"Filtering points farther than {MAX_FILTER_DISTANCE} units from their camera.")
# ----------------------------------------

PLY = o3d.geometry.PointCloud() # Use this for saving a combined cloud

last_pose = None
for i, pred1 in enumerate(predictions1):
    
    # Get raw points, colors, and the pose
    points_world = pred1["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    
    # --- START: New Filtering Logic ---
    
    # 1. Get the camera's origin (its position in the world frame)
    camera_origin = camera_pose[:3, 3]
    
    # 2. Calculate the distance of each point from the camera origin
    distances = np.linalg.norm(points_world - camera_origin, axis=1)
    
    # 3. Create a boolean mask for points *within* the distance
    mask = distances <= MAX_FILTER_DISTANCE
    
    # 4. Apply the mask to the points and colors
    filtered_points = points_world[mask]
    filtered_colors = colors[mask]
    
    # --- END: New Filtering Logic ---

    # 5. Create the point cloud from the *filtered* data
    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(filtered_points)
    pcd1.colors = o3d.utility.Vector3dVector(filtered_colors)
    
    # Get camera center for drawing the path
    camera_center = camera_pose[:3, 3]
    camera_positions.append(camera_center)
    
    # Create the camera coordinate frame visualization
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    # Add the filtered cloud and camera frame
    geometries.append(pcd1)
    geometries.append(camera_frame)

    PLY += pcd1 # Add to the combined cloud for saving

    if i == len(predictions1) - 1:
        last_pose = pred1["camera_poses"].squeeze().cpu().numpy()

# Draw the camera path
if len(camera_positions) > 1:
    line_points = o3d.utility.Vector3dVector(camera_positions)
    line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
    lines = o3d.utility.Vector2iVector(line_indices)
    
    camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    camera_path.paint_uniform_color([1, 0, 0]) # Red path
    
    geometries.append(camera_path)

# Apply the final coordinate transform
transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  
    [0,  0,  0,  1]
])

for geometry in geometries:
    geometry.transform(transform_matrix)

# 4. Display all the geometries together in one window
print(f"Displaying combined scene with {len(predictions1)} filtered point clouds...")
o3d.visualization.draw_geometries(geometries)


print(f"Saving to {output_filename}...")
PLY.transform(transform_matrix)
o3d.io.write_point_cloud(output_filename, PLY)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Filtering points farther than 10.0 units from their camera.
Displaying combined scene with 5 filtered point clouds...
Saving to /home/tong/recordings/PLYs/maritime1.ply...


True

In [4]:
o3d.visualization.draw_geometries(geometries)

In [16]:
batch2 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000250.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000300.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000350.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
    

]

views2 = load_images(batch2)

# Run inference (this will process all images in the list)
preds2 = model.infer(
    views2,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





# geometries2 = []
# camera_positions = []


# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
# geometries1.append(world_frame)

last_pose2 = None

PLY = o3d.geometry.PointCloud()


for i, pred in enumerate(preds2):
    
    points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_cam)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred["camera_poses"].squeeze().cpu().numpy()
    
    
    # pcd1.transform(camera_pose)

    # camera_center = camera_pose[:3, 3]
    # camera_positions.append(camera_center)
    T = last_pose@camera_pose
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(T)
    camera_frame.transform(transform_matrix)
    pcd.transform(T)
    pcd.transform(transform_matrix)
    
    if i>0:
        geometries.append(pcd)
        geometries.append(camera_frame)



    if i == len(preds2) - 1:
        last_pose2 = pred["camera_poses"].squeeze().cpu().numpy()  # keep batch dim
         

# if len(camera_positions) > 1:
#     # Define the points for the line set
#     line_points = o3d.utility.Vector3dVector(camera_positions)
#     # Define which points to connect (0->1, 1->2, etc.)
#     line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
#     lines = o3d.utility.Vector2iVector(line_indices)
    
#     # Create the LineSet object
#     camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
#     # Set the color of the path to red
#     camera_path.paint_uniform_color([1, 0, 0])
    
#     # Add the path to our list of things to draw
#     geometries1.append(camera_path)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  # Inverts Z
    [0,  0,  0,  1]
])

# for geometry in geometries2:
#     geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
# print(f"Displaying combined scene with {len(predictions1)} point clouds...")
# o3d.visualization.draw_geometries(geometries1)
# PLY.transform(transform_matrix)
# o3d.io.write_point_cloud(output_filename, PLY)

In [14]:
o3d.visualization.draw_geometries(geometries)

In [17]:
batch3 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000450.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000500.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000550.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000600.jpg",
    

]

views3 = load_images(batch3)

# Run inference (this will process all images in the list)
preds3 = model.infer(
    views3,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





# geometries2 = []
# camera_positions = []


# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
# geometries1.append(world_frame)

# PLY = o3d.geometry.PointCloud()
last_pose3 = None

for i, pred in enumerate(preds3):
    
    points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_cam)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred["camera_poses"].squeeze().cpu().numpy()
    
    
    # pcd1.transform(camera_pose)

    # camera_center = camera_pose[:3, 3]
    # camera_positions.append(camera_center)
    T = last_pose2@camera_pose
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(T)
    camera_frame.transform(transform_matrix)
    pcd.transform(T)
    pcd.transform(transform_matrix)
    
    if i>0:
        geometries.append(pcd)
        geometries.append(camera_frame)



    if i == len(preds2) - 1:
        last_pose3 = pred["camera_poses"].squeeze().cpu().numpy()  # keep batch dim
         

# if len(camera_positions) > 1:
#     # Define the points for the line set
#     line_points = o3d.utility.Vector3dVector(camera_positions)
#     # Define which points to connect (0->1, 1->2, etc.)
#     line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
#     lines = o3d.utility.Vector2iVector(line_indices)
    
#     # Create the LineSet object
#     camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
#     # Set the color of the path to red
#     camera_path.paint_uniform_color([1, 0, 0])
    
#     # Add the path to our list of things to draw
#     geometries1.append(camera_path)


# for geometry in geometries2:
#     geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
# print(f"Displaying combined scene with {len(predictions1)} point clouds...")
# o3d.visualization.draw_geometries(geometries1)
# PLY.transform(transform_matrix)
# o3d.io.write_point_cloud(output_filename, PLY)

In [18]:
o3d.visualization.draw_geometries(geometries)

In [ ]:
batch1 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000000.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_00000.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000100.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000150.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",

]


views1 = load_images(batch1)

# Run inference (this will process all images in the list)
preds1 = model.infer(
    views2,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)

geometries = []
last_pose = None
pcd_batch1_overlap = None # We need to save this cloud

print("Processing Batch 1...")
for i, pred1 in enumerate(preds1):
    
    # Use 'pts3d' - these are already in batch 1's world frame
    points_world = pred1["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(points_world)
    pcd1.colors = o3d.utility.Vector3dVector(colors)
    
    # Add camera pose for visualization
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    geometries.append(pcd1)
    geometries.append(camera_frame)

    if i == len(preds1) - 1:
        # This is frame_000200.jpg in World 1
        pcd_batch1_overlap = pcd1 
        last_pose = camera_pose # Save for visualization if needed

# --- BATCH 2 ---
batch2 = [
    "/home/tong/recordings/scripps924/scripps924_5/frame_000200.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000250.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000300.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000350.jpg",
    "/home/tong/recordings/scripps924/scripps924_5/frame_000400.jpg",

]

views2 = load_images(batch2)
preds2 = model.infer(views2)
# ...

pcd_batch2_overlap = None # This will be frame_000200.jpg in World 2
point_clouds_batch2 = []  # Store batch 2 clouds temporarily
camera_frames_batch2 = [] # Store batch 2 frames temporarily

print("Processing Batch 2 (in its own coordinate system)...")
for i, pred in enumerate(preds2):
    
    # Use 'pts3d' - these are in batch 2's world frame
    points_world_b2 = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
    colors_b2 = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_world_b2)
    pcd.colors = o3d.utility.Vector3dVector(colors_b2)
    
    camera_pose_b2 = pred["camera_poses"].squeeze().cpu().numpy()
    camera_frame_b2 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame_b2.transform(camera_pose_b2)

    if i == 0:
        # This is frame_000200.jpg in World 2
        pcd_batch2_overlap = pcd 
    else:
        # These are the *new* clouds (250, 300, 350, 400)
        point_clouds_batch2.append(pcd)
        camera_frames_batch2.append(camera_frame_b2)

# --- ALIGNMENT STEP (using ICP) ---

print("Aligning Batch 2 to Batch 1 using ICP...")
# Voxel downsample for faster ICP
voxel_size = 0.05 # Adjust this based on your scene's scale
source = pcd_batch2_overlap.voxel_down_sample(voxel_size)
target = pcd_batch1_overlap.voxel_down_sample(voxel_size)

# Set an initial guess (identity matrix)
trans_init = np.identity(4)

# Run ICP
# You may need to tune 'max_correspondence_distance'
reg_p2p = o3d.pipelines.registration.registration_icp(
    source, target, 0.2, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000))

# Get the transformation matrix T that maps World 2 -> World 1
T_batch2_to_batch1 = reg_p2p.transformation
print("ICP transformation found:")
print(T_batch2_to_batch1)


# --- ADD BATCH 2 GEOMETRIES (NOW TRANSFORMED) ---

print("Applying transformation to Batch 2...")
for pcd in point_clouds_batch2:
    pcd.transform(T_batch2_to_batch1)
    pcd.transform(transform_matrix) # Apply your Y/Z flip
    geometries.append(pcd)

for frame in camera_frames_batch2:
    frame.transform(T_batch2_to_batch1)
    frame.transform(transform_matrix) # Apply your Y/Z flip
    geometries.append(frame)

# Apply the Y/Z flip to batch 1 geometries
for i in range(len(predictions1) * 2): # 2 geometries (pcd, frame) per image
    geometries[i].transform(transform_matrix)

# --- VISUALIZE ---
print("Visualizing combined map...")
o3d.visualization.draw_geometries(geometries)

# # --- SAVE ---
# print(f"Saving combined PLY to {output_filename}...")
# combined_pcd = o3d.geometry.PointCloud()
# for geo in geometries:
#     if isinstance(geo, o3d.geometry.PointCloud):
#         combined_pcd += geo

# # Voxel downsample the final cloud for a reasonable file size
# final_pcd = combined_pcd.voxel_down_sample(voxel_size=0.02)
# o3d.io.write_point_cloud(output_filename, final_pcd)
# print("Done.")

Processing Batch 1...
Processing Batch 2 (in its own coordinate system)...
Aligning Batch 2 to Batch 1 using ICP...
ICP transformation found:
[[ 0.99999367  0.00335097 -0.00119958  0.07746477]
 [-0.00335421  0.9999907  -0.00271146  0.05520372]
 [ 0.00119049  0.00271546  0.9999956   0.06450189]
 [ 0.          0.          0.          1.        ]]
Applying transformation to Batch 2...
Visualizing combined map...


In [1]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
from mapanything.utils.image import preprocess_inputs

import open3d as o3d
import numpy as np
from PIL import Image

# Get inference device
# output_filename = "/home/tong/recordings/PLYs/maritime1.ply" 
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  
    [0,  0,  0,  1]
])

intrinsics = np.array([
    [980.21, 0.0, 825.18],
    [0.0, 980.21, 627.85],
    [0.0,    0.0,   1.0 ],
], dtype=np.float32)

geometries = []
poses = []
MAX_FILTER_DISTANCE = 15.0 

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


In [2]:
def initial_map(preds):
    for pred in preds:
    
        # Get raw points, colors, and the pose
        points_world = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        camera_pose = pred["camera_poses"].squeeze().cpu().numpy()

        # --- Filtering---
        camera_origin = camera_pose[:3, 3]
        distances = np.linalg.norm(points_world - camera_origin, axis=1)
        mask = distances <= MAX_FILTER_DISTANCE
        filtered_points = points_world[mask]
        filtered_colors = colors[mask]


        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(filtered_points)
        pcd.colors = o3d.utility.Vector3dVector(filtered_colors)


        camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
        camera_frame.transform(camera_pose)
        camera_frame.transform(transform_matrix)
        pcd.transform(transform_matrix)
        geometries.append(pcd)
        geometries.append(camera_frame)
        poses.append(camera_pose)

def view_process(batch):
    views = []
    for i, img in enumerate(batch):
        if i == len(batch)-1:
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "is_metric_scale": torch.tensor([True], device=device),
            })
        else:
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "camera_poses": poses[-4:][i], 
                "is_metric_scale": torch.tensor([True], device=device),
            })
    processed_views = preprocess_inputs(views)

    return processed_views


def mapping(preds):
    pred = preds[len(preds)-1]
    
    # Get raw points, colors, and the pose
    points_world = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    camera_pose = pred["camera_poses"].squeeze().cpu().numpy()

    # --- Filtering---
    camera_origin = camera_pose[:3, 3]
    distances = np.linalg.norm(points_world - camera_origin, axis=1)
    mask = distances <= MAX_FILTER_DISTANCE
    filtered_points = points_world[mask]
    filtered_colors = colors[mask]


    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(filtered_points)
    pcd.colors = o3d.utility.Vector3dVector(filtered_colors)


    camera_pose = poses[len(poses)-4]@camera_pose


    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    camera_frame.transform(transform_matrix)
    pcd.transform(camera_pose)
    pcd.transform(transform_matrix)
    geometries.append(pcd)
    geometries.append(camera_frame)
    poses.append(camera_pose)


In [3]:
# 1. Provide a list of multiple image paths
batch1 = [
    "/home/tong/recordings/moreRocks2_1/frame_0000.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0010.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0020.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0030.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0040.jpg",
]



views1 = []
for img in batch1:
    views1.append({
        "img": np.array(Image.open(img).convert("RGB")),
        "intrinsics": intrinsics,
        "is_metric_scale": torch.tensor([True], device=device),
    })

views1_1 = preprocess_inputs(views1)
# Run inference
preds1 = model.infer(
    views1_1,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

initial_map(preds1)


In [4]:
o3d.visualization.draw_geometries(geometries)

In [5]:
batch2 = [
   "/home/tong/recordings/moreRocks2_1/frame_0010.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0020.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0030.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0040.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0050.jpg",
]

views2 = view_process(batch2)
preds2 = model.infer(
    views2,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

mapping(preds2)


In [6]:
o3d.visualization.draw_geometries(geometries)

In [7]:
batch3 = [
    "/home/tong/recordings/moreRocks2_1/frame_0020.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0030.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0040.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0050.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0060.jpg",
]
views3 = view_process(batch3)
preds3 = model.infer(
    views3,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

mapping(preds3)



In [8]:
o3d.visualization.draw_geometries(geometries)

In [9]:
batch4 = [
    
     "/home/tong/recordings/moreRocks2_1/frame_0030.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0040.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0050.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0060.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0070.jpg",
]
views4 = view_process(batch4)
preds4 = model.infer(
    views4,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

mapping(preds4)

In [10]:
o3d.visualization.draw_geometries(geometries)

In [11]:
batch5 = [
    
    "/home/tong/recordings/moreRocks2_1/frame_0040.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0050.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0060.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0070.jpg",
    "/home/tong/recordings/moreRocks2_1/frame_0080.jpg",
]

views5 = view_process(batch5)
preds5 = model.infer(
    views5,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

mapping(preds5)




In [14]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
from mapanything.utils.image import preprocess_inputs

import open3d as o3d
import numpy as np
from PIL import Image

# --- Configuration ---
# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"
BASE_IMAGE_PATH = "/home/tong/recordings/moreRocks2_1/"
MAX_FILTER_DISTANCE = 15.0 
VOXEL_SIZE = 0.02 # 2cm voxel size for fusing. You can make this larger (e.g., 0.05) to get a sparser map.

# --- Model and Intrinsics ---
# Init model
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  
    [0,  0,  0,  1]
])

intrinsics = np.array([
    [980.21, 0.0, 825.18],
    [0.0, 980.21, 627.85],
    [0.0,    0.0,   1.0 ],
], dtype=np.float32)

# --- Global Lists ---
geometries = []
poses = []


# --- Helper Functions ---

def initial_map(preds):
    """Processes the very first batch to establish the world frame."""
    for pred in preds:
    
        # Get raw points, colors, and the pose
        points_world = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        camera_pose = pred["camera_poses"].squeeze().cpu().numpy()

        # --- Filtering---
        camera_origin = camera_pose[:3, 3]
        distances = np.linalg.norm(points_world - camera_origin, axis=1)
        mask = distances <= MAX_FILTER_DISTANCE
        filtered_points = points_world[mask]
        filtered_colors = colors[mask]

        # Create Open3D geometry
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(filtered_points)
        pcd.colors = o3d.utility.Vector3dVector(filtered_colors)

        camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
        camera_frame.transform(camera_pose)
        
        # Apply visualization transform
        camera_frame.transform(transform_matrix)
        pcd.transform(transform_matrix)
        
        # Store results
        geometries.append(pcd)
        geometries.append(camera_frame)
        poses.append(camera_pose)

def view_process(batch):
    """Prepares a batch of images for the model."""
    views = []
    for i, img in enumerate(batch):
        if i == len(batch)-1:
            # This is the new frame, we don't have a pose for it yet
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "is_metric_scale": torch.tensor([True], device=device),
            })
        else:
            # These are the "anchor" frames, we provide their known poses
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "camera_poses": poses[-4:][i], # Get the 4 most recent poses
                "is_metric_scale": torch.tensor([True], device=device),
            })
    processed_views = preprocess_inputs(views)

    return processed_views


def mapping(preds):
    """Processes the last frame of a prediction batch and adds it to the map."""
    pred = preds[len(preds)-1]
    
    # Get raw points, colors, and the *relative* pose
    # We use 'pts3d_cam' because it's the point cloud in the *local* camera frame
    points_local = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    camera_pose_relative = pred["camera_poses"].squeeze().cpu().numpy()

    # --- Filtering (in local camera frame) ---
    # The origin of the local camera frame is just [0,0,0]
    camera_origin_local = np.array([0.0, 0.0, 0.0])
    distances = np.linalg.norm(points_local - camera_origin_local, axis=1)
    mask = distances <= MAX_FILTER_DISTANCE
    filtered_points = points_local[mask]
    filtered_colors = colors[mask]

    # Create local Open3D geometry
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(filtered_points)
    pcd.colors = o3d.utility.Vector3dVector(filtered_colors)

    # --- Pose Calculation ---
    # This is the key: get the global pose of the *start* of the window
    # poses[-4] is the pose of frame 'i' (e.g., 10, 20, 30...)
    global_pose_of_window_start = poses[-4] 
    
    # Calculate the new global pose by chaining the transforms
    camera_pose_global = global_pose_of_window_start @ camera_pose_relative

    # Create the camera frame geometry
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose_global)

    # --- Transform to Global and Store ---
    # Transform the *local* pcd by its new *global* pose
    pcd.transform(camera_pose_global)
    
    # Apply visualization transform
    pcd.transform(transform_matrix)
    camera_frame.transform(transform_matrix)
    
    geometries.append(pcd)
    geometries.append(camera_frame)
    poses.append(camera_pose_global) # Append the new global pose


# ---
# 1. INITIALIZATION STEP
# ---
print("Processing initial batch (frames 0-40)...")
batch1 = [
    f"{BASE_IMAGE_PATH}frame_0000.jpg",
    f"{BASE_IMAGE_PATH}frame_0010.jpg",
    f"{BASE_IMAGE_PATH}frame_0020.jpg",
    f"{BASE_IMAGE_PATH}frame_0030.jpg",
    f"{BASE_IMAGE_PATH}frame_0040.jpg",
]

views1 = []
for img in batch1:
    views1.append({
        "img": np.array(Image.open(img).convert("RGB")),
        "intrinsics": intrinsics,
        "is_metric_scale": torch.tensor([True], device=device),
    })

views1_1 = preprocess_inputs(views1)
preds1 = model.infer(
    views1_1,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

initial_map(preds1)
print(f"Initial map created. Current poses: {len(poses)}")


# ---
# 2. MAPPING LOOP
# ---
# This loop replaces all your separate batch2, batch3, etc. calls
#
# We start at i = 10 (for frame_0010.jpg)
# We want the *last* frame in the last batch to be 500.
# The last frame is i + 40.
# So, the last 'i' should be 460 (460 + 40 = 500).
# range(start, stop) is exclusive, so range(10, 461, 10) is correct.

print("Starting batch processing loop...")
for i in range(10, 661, 10):
    print(f"Processing batch: frames {i} to {i+40}...")
    
    # 1. Create the batch paths
    batch = [
        f"{BASE_IMAGE_PATH}frame_{i:04d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+10:04d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+20:04d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+30:04d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+40:04d}.jpg",
    ]
    
    # 2. Process views
    # This will correctly use poses[-4:] inside the function
    views = view_process(batch)
    
    # 3. Run inference
    preds = model.infer(
        views,
        memory_efficient_inference=False,
        use_amp=True,
        amp_dtype="bf16",
        apply_mask=False,
        mask_edges=False,
        apply_confidence_mask=False,
        confidence_percentile=10,
    )
    
    # 4. Map the new frame (frame i+40)
    mapping(preds)

print("All batch processing complete.")
print(f"Total poses calculated: {len(poses)}") # Should be 5 (initial) + 46 (loop) = 51 poses

# ---
# 3. FUSING AND VISUALIZATION
# ---
print("All batches processed. Fusing point clouds...")

# Create one big point cloud to hold everything
global_pcd = o3d.geometry.PointCloud()

# Create a list for your final geometries (map + poses)
final_geometries = []

# Loop through everything you've stored
for geo in geometries:
    if isinstance(geo, o3d.geometry.PointCloud):
        # If it's a point cloud, add it to the global cloud
        global_pcd += geo
    if isinstance(geo, o3d.geometry.TriangleMesh):
        # If it's a camera pose (coordinate frame), add it to the final list
        final_geometries.append(geo)

print(f"Total points before fusing: {len(global_pcd.points)}")

# Now, merge all the overlapping points into voxels
fused_pcd = global_pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)

print(f"Total points after fusing: {len(fused_pcd.points)}")

# Add the single fused map to your final geometry list
final_geometries.insert(0, fused_pcd) # Insert at the front

print("Displaying fused map and all camera poses.")
o3d.visualization.draw_geometries(final_geometries)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Processing initial batch (frames 0-40)...
Initial map created. Current poses: 5
Starting batch processing loop...
Processing batch: frames 10 to 50...
Processing batch: frames 20 to 60...
Processing batch: frames 30 to 70...
Processing batch: frames 40 to 80...
Processing batch: frames 50 to 90...
Processing batch: frames 60 to 100...
Processing batch: frames 70 to 110...
Processing batch: frames 80 to 120...
Processing batch: frames 90 to 130...
Processing batch: frames 100 to 140...
Processing batch: frames 110 to 150...
Processing batch: frames 120 to 160...
Processing batch: frames 130 to 170...
Processing batch: frames 140 to 180...
Processing batch: frames 150 to 190...
Processing batch: frames 160 to 200...
Processing batch: frames 170 to 210...
Processing batch: frames 180 to 220...
Processing batch: frames 190 to 230...
Processing batch: frames 200 to 240...
Processing batch: frames 210 to 250...
Processing batch: frames 220 to 260...
Processing batch: frames 230 to 270...
Pro

In [17]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
from mapanything.utils.image import preprocess_inputs

import open3d as o3d
import numpy as np
from PIL import Image

# --- Configuration ---
# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"
BASE_IMAGE_PATH = "/home/tong/recordings/scripps924/scripps924_7/"
MAX_FILTER_DISTANCE = 15.0 
VOXEL_SIZE = 0.05 # 2cm voxel size for fusing. You can make this larger (e.g., 0.05) to get a sparser map.

# --- Model and Intrinsics ---
# Init model
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  
    [0,  0,  0,  1]
])

intrinsics = np.array([
    [987.46, 0.0, 830.36],
    [0.0, 980.46, 644.75],
    [0.0,    0.0,   1.0 ],
], dtype=np.float32)

# --- Global Lists ---
geometries = []
poses = []


# --- Helper Functions ---

def initial_map(preds):
    """Processes the very first batch to establish the world frame."""
    for pred in preds:
    
        # Get raw points, colors, and the pose
        points_world = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        camera_pose = pred["camera_poses"].squeeze().cpu().numpy()

        # --- Filtering---
        camera_origin = camera_pose[:3, 3]
        distances = np.linalg.norm(points_world - camera_origin, axis=1)
        mask = distances <= MAX_FILTER_DISTANCE
        filtered_points = points_world[mask]
        filtered_colors = colors[mask]

        # Create Open3D geometry
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(filtered_points)
        pcd.colors = o3d.utility.Vector3dVector(filtered_colors)

        camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
        camera_frame.transform(camera_pose)
        
        # Apply visualization transform
        camera_frame.transform(transform_matrix)
        pcd.transform(transform_matrix)
        
        # Store results
        geometries.append(pcd)
        geometries.append(camera_frame)
        poses.append(camera_pose)

def view_process(batch):
    """Prepares a batch of images for the model."""
    views = []
    for i, img in enumerate(batch):
        if i == len(batch)-1:
            # This is the new frame, we don't have a pose for it yet
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "is_metric_scale": torch.tensor([True], device=device),
            })
        else:
            # These are the "anchor" frames, we provide their known poses
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "camera_poses": poses[-4:][i], # Get the 4 most recent poses
                "is_metric_scale": torch.tensor([True], device=device),
            })
    processed_views = preprocess_inputs(views)

    return processed_views


def mapping(preds):
    """Processes the last frame of a prediction batch and adds it to the map."""
    pred = preds[len(preds)-1]
    
    # Get raw points, colors, and the *relative* pose
    # We use 'pts3d_cam' because it's the point cloud in the *local* camera frame
    points_local = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    camera_pose_relative = pred["camera_poses"].squeeze().cpu().numpy()

    # --- Filtering (in local camera frame) ---
    # The origin of the local camera frame is just [0,0,0]
    camera_origin_local = np.array([0.0, 0.0, 0.0])
    distances = np.linalg.norm(points_local - camera_origin_local, axis=1)
    mask = distances <= MAX_FILTER_DISTANCE
    filtered_points = points_local[mask]
    filtered_colors = colors[mask]

    # Create local Open3D geometry
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(filtered_points)
    pcd.colors = o3d.utility.Vector3dVector(filtered_colors)

    # --- Pose Calculation ---
    # This is the key: get the global pose of the *start* of the window
    # poses[-4] is the pose of frame 'i' (e.g., 10, 20, 30...)
    global_pose_of_window_start = poses[-4] 
    
    # Calculate the new global pose by chaining the transforms
    camera_pose_global = global_pose_of_window_start @ camera_pose_relative

    # Create the camera frame geometry
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose_global)

    # --- Transform to Global and Store ---
    # Transform the *local* pcd by its new *global* pose
    pcd.transform(camera_pose_global)
    
    # Apply visualization transform
    pcd.transform(transform_matrix)
    camera_frame.transform(transform_matrix)
    
    geometries.append(pcd)
    geometries.append(camera_frame)
    poses.append(camera_pose_global) # Append the new global pose


# ---
# 1. INITIALIZATION STEP
# ---
print("Processing initial batch (frames 0-40)...")
batch1 = [
    f"{BASE_IMAGE_PATH}frame_000000.jpg",
    f"{BASE_IMAGE_PATH}frame_000010.jpg",
    f"{BASE_IMAGE_PATH}frame_000020.jpg",
    f"{BASE_IMAGE_PATH}frame_000030.jpg",
    f"{BASE_IMAGE_PATH}frame_000040.jpg",
]

views1 = []
for img in batch1:
    views1.append({
        "img": np.array(Image.open(img).convert("RGB")),
        "intrinsics": intrinsics,
        "is_metric_scale": torch.tensor([True], device=device),
    })

views1_1 = preprocess_inputs(views1)
preds1 = model.infer(
    views1_1,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

initial_map(preds1)
print(f"Initial map created. Current poses: {len(poses)}")


# ---
# 2. MAPPING LOOP
# ---
# This loop replaces all your separate batch2, batch3, etc. calls
#
# We start at i = 10 (for frame_0010.jpg)
# We want the *last* frame in the last batch to be 500.
# The last frame is i + 40.
# So, the last 'i' should be 460 (460 + 40 = 500).
# range(start, stop) is exclusive, so range(10, 461, 10) is correct.

print("Starting batch processing loop...")
for i in range(10, 961, 10):
    print(f"Processing batch: frames {i} to {i+40}...")
    
    # 1. Create the batch paths
    batch = [
        f"{BASE_IMAGE_PATH}frame_{i:06d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+10:06d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+20:06d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+30:06d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+40:06d}.jpg",
    ]
    
    # 2. Process views
    # This will correctly use poses[-4:] inside the function
    views = view_process(batch)
    
    # 3. Run inference
    preds = model.infer(
        views,
        memory_efficient_inference=False,
        use_amp=True,
        amp_dtype="bf16",
        apply_mask=False,
        mask_edges=False,
        apply_confidence_mask=False,
        confidence_percentile=10,
    )
    
    # 4. Map the new frame (frame i+40)
    mapping(preds)

print("All batch processing complete.")
print(f"Total poses calculated: {len(poses)}") # Should be 5 (initial) + 46 (loop) = 51 poses

# ---
# 3. FUSING AND VISUALIZATION
# ---
print("All batches processed. Fusing point clouds...")

# Create one big point cloud to hold everything
global_pcd = o3d.geometry.PointCloud()

# Create a list for your final geometries (map + poses)
final_geometries = []

# Loop through everything you've stored
for geo in geometries:
    if isinstance(geo, o3d.geometry.PointCloud):
        # If it's a point cloud, add it to the global cloud
        global_pcd += geo
    if isinstance(geo, o3d.geometry.TriangleMesh):
        # If it's a camera pose (coordinate frame), add it to the final list
        final_geometries.append(geo)

print(f"Total points before fusing: {len(global_pcd.points)}")

# Now, merge all the overlapping points into voxels
fused_pcd = global_pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)

print(f"Total points after fusing: {len(fused_pcd.points)}")

# Add the single fused map to your final geometry list
final_geometries.insert(0, fused_pcd) # Insert at the front

print("Displaying fused map and all camera poses.")
o3d.visualization.draw_geometries(final_geometries)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Processing initial batch (frames 0-40)...
Initial map created. Current poses: 5
Starting batch processing loop...
Processing batch: frames 10 to 50...
Processing batch: frames 20 to 60...
Processing batch: frames 30 to 70...
Processing batch: frames 40 to 80...
Processing batch: frames 50 to 90...
Processing batch: frames 60 to 100...
Processing batch: frames 70 to 110...
Processing batch: frames 80 to 120...
Processing batch: frames 90 to 130...
Processing batch: frames 100 to 140...
Processing batch: frames 110 to 150...
Processing batch: frames 120 to 160...
Processing batch: frames 130 to 170...
Processing batch: frames 140 to 180...
Processing batch: frames 150 to 190...
Processing batch: frames 160 to 200...
Processing batch: frames 170 to 210...
Processing batch: frames 180 to 220...
Processing batch: frames 190 to 230...
Processing batch: frames 200 to 240...
Processing batch: frames 210 to 250...
Processing batch: frames 220 to 260...
Processing batch: frames 230 to 270...
Pro

In [ ]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
from mapanything.utils.image import preprocess_inputs

import open3d as o3d
import numpy as np
from PIL import Image

# --- Configuration ---
# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"
BASE_IMAGE_PATH = "/home/tong/recordings/scripps924/scripps924_7/"
MAX_FILTER_DISTANCE = 15.0 
VOXEL_SIZE = 0.1 # 2cm voxel size for fusing. You can make this larger (e.g., 0.05) to get a sparser map.

# --- Model and Intrinsics ---
# Init model
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  
    [0,  0,  0,  1]
])

intrinsics = np.array([
    [987.46, 0.0, 830.36],
    [0.0, 980.46, 644.75],
    [0.0,    0.0,   1.0 ],
], dtype=np.float32)

# --- Global Lists ---
geometries = []
poses = []


# --- Helper Functions ---

def initial_map(preds):
    """Processes the very first batch to establish the world frame."""
    for pred in preds:
    
        # Get raw points, colors, and the pose
        points_world = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        camera_pose = pred["camera_poses"].squeeze().cpu().numpy()

        # --- Filtering---
        camera_origin = camera_pose[:3, 3]
        distances = np.linalg.norm(points_world - camera_origin, axis=1)
        mask = distances <= MAX_FILTER_DISTANCE
        filtered_points = points_world[mask]
        filtered_colors = colors[mask]

        # Create Open3D geometry
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(filtered_points)
        pcd.colors = o3d.utility.Vector3dVector(filtered_colors)

        camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
        camera_frame.transform(camera_pose)
        
        # Apply visualization transform
        camera_frame.transform(transform_matrix)
        pcd.transform(transform_matrix)
        
        # Store results
        geometries.append(pcd)
        geometries.append(camera_frame)
        poses.append(camera_pose)

def view_process(batch):
    """Prepares a batch of images for the model."""
    views = []
    for i, img in enumerate(batch):
        if i == len(batch)-1:
            # This is the new frame, we don't have a pose for it yet
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "is_metric_scale": torch.tensor([True], device=device),
            })
        else:
            # These are the "anchor" frames, we provide their known poses
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "camera_poses": poses[-4:][i], # Get the 4 most recent poses
                "is_metric_scale": torch.tensor([True], device=device),
            })
    processed_views = preprocess_inputs(views)

    return processed_views


def mapping(preds):
    """Processes the last frame of a prediction batch and adds it to the map."""
    pred = preds[len(preds)-1]
    
    # Get raw points, colors, and the *relative* pose
    # We use 'pts3d_cam' because it's the point cloud in the *local* camera frame
    points_local = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    camera_pose_relative = pred["camera_poses"].squeeze().cpu().numpy()

    # --- Filtering (in local camera frame) ---
    # The origin of the local camera frame is just [0,0,0]
    camera_origin_local = np.array([0.0, 0.0, 0.0])
    distances = np.linalg.norm(points_local - camera_origin_local, axis=1)
    mask = distances <= MAX_FILTER_DISTANCE
    filtered_points = points_local[mask]
    filtered_colors = colors[mask]

    # Create local Open3D geometry
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(filtered_points)
    pcd.colors = o3d.utility.Vector3dVector(filtered_colors)

    # --- Pose Calculation ---
    # This is the key: get the global pose of the *start* of the window
    # poses[-4] is the pose of frame 'i' (e.g., 10, 20, 30...)
    global_pose_of_window_start = poses[-4] 
    
    # Calculate the new global pose by chaining the transforms
    camera_pose_global = global_pose_of_window_start @ camera_pose_relative

    # Create the camera frame geometry
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose_global)

    # --- Transform to Global and Store ---
    # Transform the *local* pcd by its new *global* pose
    pcd.transform(camera_pose_global)
    
    # Apply visualization transform
    pcd.transform(transform_matrix)
    camera_frame.transform(transform_matrix)
    
    geometries.append(pcd)
    geometries.append(camera_frame)
    poses.append(camera_pose_global) # Append the new global pose


# ---
# 1. INITIALIZATION STEP
# ---
print("Processing initial batch (frames 0-40)...")
batch1 = [
    f"{BASE_IMAGE_PATH}frame_000000.jpg",
    f"{BASE_IMAGE_PATH}frame_000010.jpg",
    f"{BASE_IMAGE_PATH}frame_000020.jpg",
    f"{BASE_IMAGE_PATH}frame_000030.jpg",
    f"{BASE_IMAGE_PATH}frame_000040.jpg",
]

views1 = []
for img in batch1:
    views1.append({
        "img": np.array(Image.open(img).convert("RGB")),
        "intrinsics": intrinsics,
        "is_metric_scale": torch.tensor([True], device=device),
    })

views1_1 = preprocess_inputs(views1)
preds1 = model.infer(
    views1_1,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

initial_map(preds1)
print(f"Initial map created. Current poses: {len(poses)}")


# ---
# 2. MAPPING LOOP
# ---
# This loop replaces all your separate batch2, batch3, etc. calls
#
# We start at i = 10 (for frame_0010.jpg)
# We want the *last* frame in the last batch to be 500.
# The last frame is i + 40.
# So, the last 'i' should be 460 (460 + 40 = 500).
# range(start, stop) is exclusive, so range(10, 461, 10) is correct.

print("Starting batch processing loop...")
for i in range(10, 961, 10):
    print(f"Processing batch: frames {i} to {i+40}...")
    
    # 1. Create the batch paths
    batch = [
        f"{BASE_IMAGE_PATH}frame_{i:06d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+10:06d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+20:06d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+30:06d}.jpg",
        f"{BASE_IMAGE_PATH}frame_{i+40:06d}.jpg",
    ]
    
    # 2. Process views
    # This will correctly use poses[-4:] inside the function
    views = view_process(batch)
    
    # 3. Run inference
    preds = model.infer(
        views,
        memory_efficient_inference=False,
        use_amp=True,
        amp_dtype="bf16",
        apply_mask=False,
        mask_edges=False,
        apply_confidence_mask=False,
        confidence_percentile=10,
    )
    
    # 4. Map the new frame (frame i+40)
    mapping(preds)

print("All batch processing complete.")
print(f"Total poses calculated: {len(poses)}") # Should be 5 (initial) + 46 (loop) = 51 poses

# ---
# 3. FUSING AND VISUALIZATION
# ---
print("All batches processed. Fusing point clouds...")

# Create one big point cloud to hold everything
global_pcd = o3d.geometry.PointCloud()

# Create a list for your final geometries (map + poses)
final_geometries = []

# Loop through everything you've stored
for geo in geometries:
    if isinstance(geo, o3d.geometry.PointCloud):
        # If it's a point cloud, add it to the global cloud
        global_pcd += geo
    if isinstance(geo, o3d.geometry.TriangleMesh):
        # If it's a camera pose (coordinate frame), add it to the final list
        final_geometries.append(geo)

print(f"Total points before fusing: {len(global_pcd.points)}")

# Now, merge all the overlapping points into voxels
fused_pcd = global_pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)

print(f"Total points after fusing: {len(fused_pcd.points)}")

# Add the single fused map to your final geometry list
final_geometries.insert(0, fused_pcd) # Insert at the front

print("Displaying fused map and all camera poses.")
o3d.visualization.draw_geometries(final_geometries)

In [3]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
from mapanything.utils.image import preprocess_inputs

import open3d as o3d
import numpy as np
from PIL import Image

# --- Configuration ---
# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"
BASE_IMAGE_PATH = "/home/tong/recordings/scripps924/scripps924_5/"
MAX_FILTER_DISTANCE = 15.0 
VOXEL_SIZE = 0.1# 2cm voxel size for fusing.

# --- !! NEW CONFIGURABLE VARIABLES !! ---
KEYFRAME_STEP = 60       # The gap between keyframes (e.g., 10, 20, 30)
END_FRAME = 1300         # The last frame number you want to process
NUM_VIEWS_PER_BATCH = 5  
# ----------------------------------------

# --- Model and Intrinsics ---
# Init model
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  
    [0,  0,  0,  1]
])

intrinsics = np.array([
    [987.46, 0.0, 830.36],
    [0.0, 987.46, 644.75],
    [0.0,    0.0,   1.0 ],
], dtype=np.float32)

# --- Global Lists ---
geometries = []
poses = []


# --- Helper Functions (No changes needed) ---

def initial_map(preds):
    """Processes the very first batch to establish the world frame."""
    for pred in preds:
    
        # Get raw points, colors, and the pose
        points_world = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        camera_pose = pred["camera_poses"].squeeze().cpu().numpy()

        # --- Filtering---
        camera_origin = camera_pose[:3, 3]
        distances = np.linalg.norm(points_world - camera_origin, axis=1)
        mask = distances <= MAX_FILTER_DISTANCE
        filtered_points = points_world[mask]
        filtered_colors = colors[mask]

        # Create Open3D geometry
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(filtered_points)
        pcd.colors = o3d.utility.Vector3dVector(filtered_colors)

        camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
        camera_frame.transform(camera_pose)
        
        # Apply visualization transform
        camera_frame.transform(transform_matrix)
        pcd.transform(transform_matrix)
        
        # Store results
        geometries.append(pcd)
        geometries.append(camera_frame)
        poses.append(camera_pose)

def view_process(batch):
    """Prepares a batch of images for the model."""
    views = []
    for i, img in enumerate(batch):
        if i == len(batch)-1:
            # This is the new frame, we don't have a pose for it yet
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "is_metric_scale": torch.tensor([True], device=device),
            })
        else:
            # These are the "anchor" frames, we provide their known poses
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "camera_poses": poses[-4:][i], # Get the 4 most recent poses
                "is_metric_scale": torch.tensor([True], device=device),
            })
    processed_views = preprocess_inputs(views)

    return processed_views


def mapping(preds):
    """Processes the last frame of a prediction batch and adds it to the map."""
    pred = preds[len(preds)-1]
    
    # Get raw points, colors, and the *relative* pose
    # We use 'pts3d_cam' because it's the point cloud in the *local* camera frame
    points_local = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    camera_pose_relative = pred["camera_poses"].squeeze().cpu().numpy()

    # --- Filtering (in local camera frame) ---
    # The origin of the local camera frame is just [0,0,0]
    camera_origin_local = np.array([0.0, 0.0, 0.0])
    distances = np.linalg.norm(points_local - camera_origin_local, axis=1)
    mask = distances <= MAX_FILTER_DISTANCE
    filtered_points = points_local[mask]
    filtered_colors = colors[mask]

    # Create local Open3D geometry
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(filtered_points)
    pcd.colors = o3d.utility.Vector3dVector(filtered_colors)

    # --- Pose Calculation ---
    # This is the key: get the global pose of the *start* of the window
    # poses[-4] is the pose of frame 'i' (e.g., 20, 40, 60...)
    global_pose_of_window_start = poses[-4] 
    
    # Calculate the new global pose by chaining the transforms
    camera_pose_global = global_pose_of_window_start @ camera_pose_relative

    # Create the camera frame geometry
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose_global)

    # --- Transform to Global and Store ---
    # Transform the *local* pcd by its new *global* pose
    pcd.transform(camera_pose_global)
    
    # Apply visualization transform
    pcd.transform(transform_matrix)
    camera_frame.transform(transform_matrix)
    
    geometries.append(pcd)
    geometries.append(camera_frame)
    poses.append(camera_pose_global) # Append the new global pose


# ---
# 1. INITIALIZATION STEP (NOW DYNAMIC)
# ---
# This calculates the span of the first window, e.g., 4 * 20 = 80
window_span = (NUM_VIEWS_PER_BATCH - 1) * KEYFRAME_STEP 

print(f"Processing initial batch (frames 0 to {window_span}, step {KEYFRAME_STEP})...")
batch1 = []
for i in range(NUM_VIEWS_PER_BATCH):
    frame_num = i * KEYFRAME_STEP
    batch1.append(f"{BASE_IMAGE_PATH}frame_{frame_num:06d}.jpg")

# --- This part is the same ---
views1 = []
for img in batch1:
    views1.append({
        "img": np.array(Image.open(img).convert("RGB")),
        "intrinsics": intrinsics,
        "is_metric_scale": torch.tensor([True], device=device),
    })

views1_1 = preprocess_inputs(views1)
preds1 = model.infer(
    views1_1,
    memory_efficient_inference=False,
    use_amp=True,
    amp_dtype="bf16",
    apply_mask=False,
    mask_edges=False,
    apply_confidence_mask=False,
    confidence_percentile=10,
)

initial_map(preds1)
print(f"Initial map created. Current poses: {len(poses)}")


# ---
# 2. MAPPING LOOP (NOW DYNAMIC)
# ---
#
# This loop is now controlled by your variables at the top.
# It starts at the first keyframe (e.g., 20)
# It stops when the *last* frame of the batch (i + window_span) would exceed END_FRAME
# e.g., for END_FRAME=600, window_span=80, the last 'i' is 520. (520+80=600)
# The range's stop value is exclusive, so (END_FRAME - window_span) + 1 is correct.

print(f"Starting batch processing loop (step {KEYFRAME_STEP})...")
start_frame = KEYFRAME_STEP
stop_frame = (END_FRAME - window_span) + 1
step = KEYFRAME_STEP

for i in range(start_frame, stop_frame, step):
    print(f"Processing batch: frames {i} to {i + window_span}...")
    
    # 1. Create the batch paths dynamically
    batch = []
    for j in range(NUM_VIEWS_PER_BATCH):
        frame_num = i + (j * KEYFRAME_STEP)
        batch.append(f"{BASE_IMAGE_PATH}frame_{frame_num:06d}.jpg")

    # 2. Process views
    # This will correctly use poses[-4:] inside the function
    views = view_process(batch)
    
    # 3. Run inference
    preds = model.infer(
        views,
        memory_efficient_inference=False,
        use_amp=True,
        amp_dtype="bf16",
        apply_mask=False,
        mask_edges=False,
        apply_confidence_mask=False,
        confidence_percentile=10,
    )
    
    # 4. Map the new frame (frame i + window_span)
    mapping(preds)

print("All batch processing complete.")
# This calculates the number of loop iterations
num_loop_poses = (END_FRAME - window_span) // KEYFRAME_STEP
print(f"Total poses calculated: {len(poses)}") 
print(f"({NUM_VIEWS_PER_BATCH} initial + {num_loop_poses} loop = {NUM_VIEWS_PER_BATCH + num_loop_poses} poses)")


# ---
# 3. FUSING AND VISUALIZATION (Using your variable name)
# ---
print("All batches processed. Fusing point clouds...")

# Create one big point cloud to hold everything
global_pcd = o3d.geometry.PointCloud()

# Create a list for your final geometries (map + poses)
scripps924_5 = [] # Using your variable name

# Loop through everything you've stored
for geo in geometries:
    if isinstance(geo, o3d.geometry.PointCloud):
        # If it's a point cloud, add it to the global cloud
        global_pcd += geo
    if isinstance(geo, o3d.geometry.TriangleMesh):
        # If it's a camera pose (coordinate frame), add it to the final list
        scripps924_5.append(geo)

print(f"Total points before fusing: {len(global_pcd.points)}")

# Now, merge all the overlapping points into voxels
fused_pcd = global_pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)

print(f"Total points after fusing: {len(fused_pcd.points)}")

# Add the single fused map to your final geometry list
scripps924_5.insert(0, fused_pcd) # Insert at the front


Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Processing initial batch (frames 0 to 240, step 60)...
Initial map created. Current poses: 5
Starting batch processing loop (step 60)...
Processing batch: frames 60 to 300...
Processing batch: frames 120 to 360...
Processing batch: frames 180 to 420...
Processing batch: frames 240 to 480...
Processing batch: frames 300 to 540...
Processing batch: frames 360 to 600...
Processing batch: frames 420 to 660...
Processing batch: frames 480 to 720...
Processing batch: frames 540 to 780...
Processing batch: frames 600 to 840...
Processing batch: frames 660 to 900...
Processing batch: frames 720 to 960...
Processing batch: frames 780 to 1020...
Processing batch: frames 840 to 1080...
Processing batch: frames 900 to 1140...
Processing batch: frames 960 to 1200...
Processing batch: frames 1020 to 1260...
All batch processing complete.
Total poses calculated: 22
(5 initial + 17 loop = 22 poses)
All batches processed. Fusing point clouds...
Total points before fusing: 2902282
Total points after fus

In [4]:

o3d.visualization.draw_geometries(scripps924_5)